# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides a step-by-step guide for loading, exploring, and analyzing the [FAIR^2 Mass Spectrometry](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Access metadata fields
print("Dataset name:", dataset.metadata.name)
print("Description:", dataset.metadata.description)
print("Published:", getattr(dataset.metadata, 'datePublished', 'N/A'))
print("Cite as:", getattr(dataset.metadata, 'citeAs', 'N/A'))

## 2. Data Overview
Review available record sets, fields, and their IDs.

### List all record sets and their fields using their `@id`.

For each record set, we display:
- The record set `@id`
- Its title (if available)
- Its available fields & columns and their `@id` and names

In [ ]:
# Get all record set IDs
record_sets = dataset.record_sets
print(f"Number of record sets: {len(record_sets)}")

for record_set in record_sets:
    print(f"\nRecord set @id: {record_set.id}")
    print(f"  Name: {getattr(record_set, 'name', 'N/A')}")
    fields = getattr(record_set, 'fields', [])
    print(f"  Fields in this record set:")
    for field in fields:
        print(f"    - Field @id: {field.id:40} | Name: {getattr(field, 'name', 'N/A')}")
    columns = getattr(record_set, 'columns', [])
    if columns:
        print("  Columns:")
        for col in columns:
            print(f"    - Column @id: {col.id:40} | Name: {getattr(col, 'name', 'N/A')}")

## 3. Data Extraction

Load data from a specific record set into a DataFrame for analysis.

Let's extract all tabular record sets, print their columns, and load the first record set into a DataFrame.

In [ ]:
# Prepare list of record set @id (using @id for explicit referencing)
record_set_ids = [rs.id for rs in dataset.record_sets]
dataframes = {}

for rsid in record_set_ids:
    # Load all records for each record set
    records = list(dataset.records(record_set=rsid))
    dataframes[rsid] = pd.DataFrame(records)
    print(f"\nRecord set @id: {rsid}")
    print("Fields/Columns:", dataframes[rsid].columns.tolist())

# For demonstration, pick the first record set
selected_record_set_id = record_set_ids[0]
print(f"\nWorking with record set @id: {selected_record_set_id}")
dataframes[selected_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)

Let's demonstrate common EDA steps such as filtering records, normalizing numeric fields, and grouping data. All fields are referenced by their `@id`.

- Select a numeric field and a group/category field for demonstration.

In [ ]:
# List the available columns to choose appropriate field @ids
print("Available columns:")
print(dataframes[selected_record_set_id].columns.tolist())

# For illustration, pick a numeric field
# Replace these with the actual field @id for numeric and grouping fields after inspecting above
numeric_field_id = dataframes[selected_record_set_id].columns[1]  # Example: the second column

threshold = 10
df = dataframes[selected_record_set_id]

# Filter rows where numeric field > threshold
if pd.api.types.is_numeric_dtype(df[numeric_field_id]):
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalize the numeric field
    filtered_df[numeric_field_id + '_normalized'] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    )
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, numeric_field_id + '_normalized']].head())

    # Try grouping by the first text/string column (if exists)
    text_fields = [c for c in df.columns if pd.api.types.is_object_dtype(df[c]) and c != numeric_field_id]
    if text_fields:
        group_field_id = text_fields[0]
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
        print(f"\nGrouped data by {group_field_id} (mean of {numeric_field_id}):")
        print(grouped_df.head())
else:
    print(f"Selected numeric field '{numeric_field_id}' is not numeric. Please adjust the numeric_field_id.")

## 5. Visualization

Visualize the distribution of the numeric field and potential relationships with a category/group field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field
if pd.api.types.is_numeric_dtype(df[numeric_field_id]):
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # Boxplot by group field (if assigned above)
    if 'group_field_id' in locals():
        plt.figure(figsize=(10,4))
        sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=30, ha='right')
        plt.show()
else:
    print(f"{numeric_field_id} is not numeric or not present for plotting.")

## 6. Conclusion

In this notebook, we loaded the FAIR^2 protein abundance dataset defined by a Croissant schema, explored the available record sets and fields (using `@id` for reproducibility), extracted tabular data, performed filtering and normalization, grouped by relevant columns, and visualized the distributions. This workflow can be adapted for similar FAIR-compatible datasets for robust data exploration and analysis.